In [ ]:
!pip install --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv


In [ ]:
import glob
import os
import shutil
import subprocess
import sys
import time

def find_model_dir(keyword):
    """Search /kaggle/input for a directory containing model weight files
    whose path includes `keyword` -- robust to whatever the actual mount
    convention turns out to be."""
    candidates = []
    for path in glob.glob("/kaggle/input/**/config.json", recursive=True):
        if keyword.lower() in path.lower():
            candidates.append(os.path.dirname(path))
    if not candidates:
        return None
    return sorted(candidates)[-1]


if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    _t0 = time.time()

    def _el():
        return f"{time.time() - _t0:6.1f}s"

    # Write a placeholder submission before anything risky runs -- same
    # insurance-against-total-setup-failure rationale as the sibling JEPA
    # project's notebook: if setup below throws before main.py ever plays
    # a game, this is the only thing standing between the run and no
    # submission file at all.
    import pandas as pd
    pd.DataFrame(
        data=[["1_0", "1", True, 1]],
        columns=["row_id", "game_id", "end_of_game", "score"],
    ).to_parquet("/kaggle/working/submission.parquet", index=False)
    print(f"[{_el()}] === step: wrote placeholder submission.parquet ===", flush=True)

    # 600s wait in background matching official starter and proven latency
    print(f"[{_el()}] === step: starting gateway wait in background ===", flush=True)
    gateway_proc = subprocess.Popen(
        ["curl", "--fail", "--retry", "999", "--retry-all-errors",
         "--retry-delay", "2", "--retry-max-time", "600",
         "http://gateway:8001/api/games"],
    )

    CODER_MODEL_DIR = find_model_dir("qwen3-coder") or find_model_dir("qwen")
    ACTION_MODEL_DIR = find_model_dir("gemma-3-12b-it") or find_model_dir("gemma")
    print(f"[{_el()}] CODER_MODEL_DIR = {CODER_MODEL_DIR}", flush=True)
    print(f"[{_el()}] ACTION_MODEL_DIR = {ACTION_MODEL_DIR}", flush=True)
    assert CODER_MODEL_DIR, "could not locate coder model directory"
    assert ACTION_MODEL_DIR, "could not locate action_head model directory"

    print(f"[{_el()}] === step: copying competition harness ===", flush=True)
    _COMP = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3"
    shutil.copytree(
        f"{_COMP}/ARC-AGI-3-Agents",
        "/kaggle/working/ARC-AGI-3-Agents",
        ignore=shutil.ignore_patterns(".git"),
    )
    shutil.copytree(
        f"{_COMP}/environment_files",
        "/kaggle/working/ARC-AGI-3-Agents/environment_files",
    )

    print(f"[{_el()}] === step: copying llm_engine package + agent ===", flush=True)
    _DATASET = "/kaggle/input/datasets/calamitychasm/llm-world-engine-agent"
    shutil.copytree(f"{_DATASET}/llm_engine", "/kaggle/working/llm_engine")
    shutil.copy(
        f"{_DATASET}/code_world_agent.py",
        "/kaggle/working/ARC-AGI-3-Agents/agents/templates/code_world_agent.py",
    )

    print(f"[{_el()}] === step: verifying copied files exist ===", flush=True)
    for p in [
        "/kaggle/working/llm_engine/drafting.py",
        "/kaggle/working/llm_engine/world_model.py",
        "/kaggle/working/llm_engine/action_head.py",
        "/kaggle/working/ARC-AGI-3-Agents/agents/templates/code_world_agent.py",
        "/kaggle/working/ARC-AGI-3-Agents/environment_files",
    ]:
        assert os.path.exists(p), f"missing expected path: {p}"
    print(f"[{_el()}] all expected files present", flush=True)

    print(f"[{_el()}] === step: writing agents/__init__.py ===", flush=True)
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write(
            "from typing import Type, cast\n"
            "from dotenv import load_dotenv\n"
            "from .agent import Agent, Playback\n"
            "from .swarm import Swarm\n"
            "from .templates.random_agent import Random\n"
            "from .templates.code_world_agent import CodeWorldAgent\n"
            "\n"
            "load_dotenv()\n"
            "\n"
            "AVAILABLE_AGENTS: dict[str, Type[Agent]] = {\n"
            "    \"random\": Random,\n"
            "    \"codeworldagent\": CodeWorldAgent,\n"
            "}\n"
        )

    print(f"[{_el()}] === step: sanity-importing our agent before running main.py ===", flush=True)
    sys.path.insert(0, "/kaggle/working/ARC-AGI-3-Agents")
    sys.path.insert(0, "/kaggle/working")
    from agents.templates.code_world_agent import CodeWorldAgent  # noqa: F401
    print(f"[{_el()}] agent import OK", flush=True)

    print(f"[{_el()}] === step: writing .env ===", flush=True)
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write(
            "SCHEME=http\n"
            "HOST=gateway\n"
            "PORT=8001\n"
            "ARC_API_KEY=test-key-123\n"
            "ARC_BASE_URL=http://gateway:8001/\n"
            "OPERATION_MODE=online\n"
            "ENVIRONMENTS_DIR=\n"
            "RECORDINGS_DIR=/kaggle/working/server_recording\n"
        )

    print(f"[{_el()}] === step: joining background gateway-wait ===", flush=True)
    gw_rc = gateway_proc.wait()
    print(f"[{_el()}] gateway curl exited with code {gw_rc}", flush=True)
    if gw_rc != 0:
        print(f"[{_el()}] WARNING: gateway curl exited with {gw_rc}, proceeding to main.py", flush=True)

    print(f"[{_el()}] === step: running agent ===", flush=True)
    run_env = {
        **os.environ,
        "MPLBACKEND": "agg",
        "LLM_BACKEND": "transformers",
        "CODER_MODEL_DIR": CODER_MODEL_DIR,
        "ACTION_MODEL_DIR": ACTION_MODEL_DIR,
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    }
    result = subprocess.run(
        [sys.executable, "main.py", "--agent", "codeworldagent"],
        cwd="/kaggle/working/ARC-AGI-3-Agents",
        env=run_env,
    )
    print(f"[{_el()}] === main.py exited with code {result.returncode} ===", flush=True)


In [ ]:
# Non-rerun mode: produce a dummy submission so this notebook "runs
# successfully" during editing without needing the gateway/GPU.
import pandas as pd

if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
